# 1. Kết nối SQlite, tạo bảng student và bảng course

In [ ]:
import sqlite3
import pandas as pd

In [ ]:
# Kết nối SQLite
conn = sqlite3.connect("students.db")
cursor = conn.cursor()

In [ ]:
cursor.execute("DROP TABLE IF EXISTS student;")
cursor.execute("DROP TABLE IF EXISTS course;")

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
);
""")

# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
);
""")

# Lưu thay đổi vào database
conn.commit()

In [ ]:
# Dữ liệu cho bảng student
student_data = [
    (1, "Nguyen Minh Hoang", "May Tinh", 12, 6.7),
    (2, "Tran Thi Lan", "Kinh Te", 34, 9.2),
    (3, "Pham Van Nam", "Toan Tin", None, 7.9),
    (4, "Le Thanh Huyen", "Toan Tin", 20, 7.2),
    (5, "Vu Quoc Anh", "May Tinh", 24, 8.0),
    (6, "Dang Thuy Linh", "May Tinh", 24, 5.5),
    (7, "Bui Tien Dung", "Kinh Te", 34, 9.2),
    (8, "Ho Ngoc Mai", "Toan Tin", 20, 8.8),
    (9, "Duong Huu Phuc", "Kinh Te", None, 7.2),
    (10, "Cao Thi Hanh", "May Tinh", None, 7.0)
]

# Dữ liệu cho bảng course
course_data = [
    (12, "Giai tich"),
    (34, "Thong ke"),
    (26, "Tin hoc")
]

# Chèn dữ liệu vào bảng student
cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?);", student_data)

# Chèn dữ liệu vào bảng course
cursor.executemany("INSERT INTO course VALUES (?, ?);", course_data)

conn.commit()


In [ ]:
# Đọc dữ liệu từ bảng student
student_df = pd.read_sql_query("SELECT * FROM student;", conn)
print("Bảng Student:")
display(student_df)

# Đọc dữ liệu từ bảng course
course_df = pd.read_sql_query("SELECT * FROM course;", conn)
print("Bảng Course:")
display(course_df)

Bảng Student:


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


Bảng Course:


,id,course_name
0,12,Giai tich
1,26,Tin hoc
2,34,Thong ke


**Nhận xét **

-Trong bảng Student:

+ Số lượng: 10 dòng, 5 cột.
+ Bảng Student có các cột: student_id (ID sinh viên, kiểu số nguyên), name (tên sinh viên, kiểu chuỗi), class (lớp, kiểu chuỗi), course_id (ID môn học, kiểu số thực, có giá trị thiếu), score (điểm, kiểu số thực).
+ Cột course_id có 3 giá trị bị thiếu (NaN) ở các sinh viên có student_id là 3, 9 và 10, điều này có thể gây khó khăn khi liên kết với bảng Course.
Ngoài ra, tồn tại các giá trị course_id (20, 24) không có trong bảng Course, dẫn đến dữ liệu không đồng nhất giữa hai bảng.

-Trong bảng Course:

+ Số lượng: 3 dòng, 2 cột.
+ Bảng Course có các cột: id (ID môn học, kiểu số nguyên) và course_name (tên môn học, kiểu chuỗi).
+ Số lượng môn học còn hạn chế, chưa bao phủ hết các course_id xuất hiện trong bảng Student.

#Câu 1. Hãy kết nối hai bảng trên theo những cách sau:
- Sử dụng tích Decartes.
- Sử dụng JOIN: INNER JOIN, LEFT JOIN, RIGHT JOIN, FULL OUTER JOIN.

## a) Sử dụng tích Decartes.

In [ ]:
import pandas as pd

print("Tích Descartes (CROSS JOIN):")

cursor.execute('''
SELECT
    s.student_id,
    s.name,
    s.class,
    s.course_id,
    s.score,
    c.id AS course_id_ref,
    c.course_name
FROM student s
CROSS JOIN course c;
''')

df_decartes = pd.DataFrame(cursor.fetchall(), columns=[
    'student_id', 'name', 'class', 'course_id', 'score',
    'course_id_ref', 'course_name'
])

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(df_decartes)
print("\n")

Tích Descartes (CROSS JOIN):
    student_id               name     class  course_id  score  course_id_ref course_name
0            1  Nguyen Minh Hoang  May Tinh       12.0    6.7             12   Giai tich
1            1  Nguyen Minh Hoang  May Tinh       12.0    6.7             26     Tin hoc
2            1  Nguyen Minh Hoang  May Tinh       12.0    6.7             34    Thong ke
3            2       Tran Thi Lan   Kinh Te       34.0    9.2             12   Giai tich
4            2       Tran Thi Lan   Kinh Te       34.0    9.2             26     Tin hoc
5            2       Tran Thi Lan   Kinh Te       34.0    9.2             34    Thong ke
6            3       Pham Van Nam  Toan Tin        NaN    7.9             12   Giai tich
7            3       Pham Van Nam  Toan Tin        NaN    7.9             26     Tin hoc
8            3       Pham Van Nam  Toan Tin        NaN    7.9             34    Thong ke
9            4     Le Thanh Huyen  Toan Tin       20.0    7.2             12   Gi

**Nhận xét**

Kết quả thu được là phép tích Descartes giữa hai bảng student và course.

Mỗi sinh viên trong bảng student được kết hợp với tất cả các môn học trong bảng course,
không phụ thuộc vào giá trị course_id.

Do bảng student có 10 bản ghi và bảng course có 3 bản ghi nên kết quả thu được là 30 bản ghi.

Có thể thấy rằng:
- Một sinh viên xuất hiện nhiều lần, tương ứng với từng môn học.
- Các giá trị course_id của sinh viên không ảnh hưởng đến kết quả ghép.
- Ngay cả những sinh viên có course_id bị thiếu (NULL) hoặc không tồn tại trong bảng course
  vẫn được kết hợp với tất cả các môn học.

Phép tích Descartes không phù hợp để lấy dữ liệu liên kết thực tế giữa hai bảng,
mà chủ yếu dùng để minh họa hoặc tạo tất cả các tổ hợp có thể xảy ra.

##b) Sử dụng JOIN: INNER JOIN, LEFT JOIN, RIGHT JOIN, FULL OUTER JOIN.

### INNER JOIN

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("INNER JOIN:")

cursor.execute('''
SELECT
    s.student_id,
    s.name,
    s.class,
    s.course_id,
    s.score,
    c.course_name
FROM student s
INNER JOIN course c
    ON s.course_id = c.id;
''')

df_inner = pd.DataFrame(cursor.fetchall(), columns=[
    'student_id', 'name', 'class', 'course_id', 'score', 'course_name'
])

print(df_inner)
print("\n")

INNER JOIN:
   student_id               name     class  course_id  score course_name
0           1  Nguyen Minh Hoang  May Tinh         12    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te         34    9.2    Thong ke
2           7      Bui Tien Dung   Kinh Te         34    9.2    Thong ke




**Nhận xét**

- Bảng chỉ hiển thị các dòng có giá trị khớp giữa course_id của bảng student và id của bảng course.
- Số lượng sinh viên xuất hiện ít, cho thấy nhiều sinh viên có course_id bị thiếu hoặc không tồn tại trong bảng course.

### LEFT JOIN

In [ ]:
print("LEFT JOIN:")

cursor.execute('''
SELECT
    s.student_id,
    s.name,
    s.class,
    s.course_id,
    s.score,
    c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id;
''')

df_left = pd.DataFrame(cursor.fetchall(), columns=[
    'student_id', 'name', 'class', 'course_id', 'score', 'course_name'
])

print(df_left)
print("\n")

LEFT JOIN:
   student_id               name     class  course_id  score course_name
0           1  Nguyen Minh Hoang  May Tinh       12.0    6.7   Giai tich
1           2       Tran Thi Lan   Kinh Te       34.0    9.2    Thong ke
2           3       Pham Van Nam  Toan Tin        NaN    7.9        None
3           4     Le Thanh Huyen  Toan Tin       20.0    7.2        None
4           5        Vu Quoc Anh  May Tinh       24.0    8.0        None
5           6     Dang Thuy Linh  May Tinh       24.0    5.5        None
6           7      Bui Tien Dung   Kinh Te       34.0    9.2    Thong ke
7           8        Ho Ngoc Mai  Toan Tin       20.0    8.8        None
8           9     Duong Huu Phuc   Kinh Te        NaN    7.2        None
9          10       Cao Thi Hanh  May Tinh        NaN    7.0        None




**Nhận xét**

- Bảng hiển thị toàn bộ sinh viên, kể cả những sinh viên không có course_id hợp lệ.
- Các sinh viên không khớp với bảng course sẽ có course_name = NULL

### RIGHT JOIN

In [ ]:
print("RIGHT JOIN:")

cursor.execute('''
SELECT
    s.student_id,
    s.name,
    s.class,
    s.course_id,
    s.score,
    c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id;
''')

df_right = pd.DataFrame(cursor.fetchall(), columns=[
    'student_id', 'name', 'class', 'course_id', 'score', 'course_name'
])

print(df_right)
print("\n")

RIGHT JOIN:
   student_id               name     class  course_id  score course_name
0         1.0  Nguyen Minh Hoang  May Tinh       12.0    6.7   Giai tich
1         NaN               None      None        NaN    NaN     Tin hoc
2         7.0      Bui Tien Dung   Kinh Te       34.0    9.2    Thong ke
3         2.0       Tran Thi Lan   Kinh Te       34.0    9.2    Thong ke




**Nhận xét**

- Kết quả giữ lại toàn bộ các môn học trong bảng course, kể cả khi không có sinh viên đăng ký.
- Các dòng có đầy đủ thông tin sinh viên cho thấy course_id của sinh viên khớp với id của bảng course.
- Có xuất hiện dòng mà các cột student (student_id, name, class, score) = ,
  điều này cho thấy tồn tại môn học không có sinh viên nào tham gia.
- So với INNER JOIN, RIGHT JOIN giữ được nhiều dữ liệu hơn vì không loại bỏ các môn học không có sinh viên.

### FULL OUTER JOIN.

In [ ]:
print("FULL OUTER JOIN:")

cursor.execute('''
SELECT
    s.student_id,
    s.name,
    s.class,
    c.id AS course_id_ref,
    s.course_id AS student_course_id,
    s.score,
    c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id

UNION ALL

SELECT
    s.student_id,
    s.name,
    s.class,
    c.id AS course_id_ref,
    s.course_id AS student_course_id,
    s.score,
    c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
WHERE s.student_id IS NULL;
''')

result = cursor.fetchall()

df_full = pd.DataFrame(result, columns=[
    'student_id', 'name', 'class', 'course_id_ref', 'student_course_id', 'score', 'course_name'
])

print(df_full)

FULL OUTER JOIN:
    student_id               name     class  course_id_ref  student_course_id  score course_name
0          1.0  Nguyen Minh Hoang  May Tinh           12.0               12.0    6.7   Giai tich
1          2.0       Tran Thi Lan   Kinh Te           34.0               34.0    9.2    Thong ke
2          3.0       Pham Van Nam  Toan Tin            NaN                NaN    7.9        None
3          4.0     Le Thanh Huyen  Toan Tin            NaN               20.0    7.2        None
4          5.0        Vu Quoc Anh  May Tinh            NaN               24.0    8.0        None
5          6.0     Dang Thuy Linh  May Tinh            NaN               24.0    5.5        None
6          7.0      Bui Tien Dung   Kinh Te           34.0               34.0    9.2    Thong ke
7          8.0        Ho Ngoc Mai  Toan Tin            NaN               20.0    8.8        None
8          9.0     Duong Huu Phuc   Kinh Te            NaN                NaN    7.2        None
9         10.

**Nhận xét**

- Bảng hiển thị toàn bộ dữ liệu của cả hai bảng student và course.
- Các sinh viên không có course_id hợp lệ vẫn xuất hiện, với course_name = NULL.
- Môn học không có sinh viên đăng ký cũng xuất hiện, với thông tin sinh viên = NULL.
- Cụ thể, môn "Tin hoc" xuất hiện với toàn bộ thông tin sinh viên = NULL, cho thấy không có sinh viên nào đăng ký môn học này.
- Kết quả bao gồm cả dữ liệu khớp và không khớp giữa hai bảng.

# Câu 2. Hãy cập nhật những giá trị course_id còn thiếu trong bảng student bằng câu lệnh SQL, trong đó các giá trị được điền là những giá trị nằm trong bảng course và loại bỏ những bản ghi tham gia những môn học không tồn tại bảng course. Sau đó hãy cho biết:

a. Tổng số sinh viên, điểm trung bình của từng lớp.

b. Tổng số sinh viên, điểm trung bình của từng môn học.

c. Phân loại thi đua theo số điểm của từng môn học biết:

-   i. Điểm TB ≥  9.0: Xuất sắc.
-   ii. 6.0 ≤ Điểm TB ≤ 8.9: Tốt.
-  iii. Điểm TB < 6.0: Kém.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# =========================
# 1. Cập nhật course_id còn thiếu
# =========================
print("Cập nhật các giá trị course_id còn thiếu:")

cursor.execute('''
UPDATE student
SET course_id = (
    SELECT id
    FROM course
    ORDER BY RANDOM()
    LIMIT 1
)
WHERE course_id IS NULL;
''')
conn.commit()

# Hiển thị dữ liệu sau khi cập nhật
cursor.execute('''
SELECT *
FROM student;
''')

df_updated = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score']
)

print(df_updated)
print("\n")


# =========================
# 2. Xóa các bản ghi có course_id không hợp lệ
# =========================
print("Loại bỏ các bản ghi có course_id không tồn tại trong bảng course:")

cursor.execute('''
DELETE FROM student
WHERE course_id NOT IN (
    SELECT id
    FROM course
);
''')
conn.commit()

# Hiển thị dữ liệu sau khi xóa
cursor.execute('''
SELECT *
FROM student;
''')

df_cleaned = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score']
)

print(df_cleaned)
print("\n")


# =========================
# 3. Kiểm tra lại dữ liệu sau xử lý
# =========================
print("Bảng student sau khi cập nhật và làm sạch dữ liệu:")

cursor.execute('''
SELECT *
FROM student
ORDER BY student_id;
''')

df_final = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score']
)

print(df_final)

Cập nhật các giá trị course_id còn thiếu:
   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh         12    6.7
1           2       Tran Thi Lan   Kinh Te         34    9.2
2           3       Pham Van Nam  Toan Tin         26    7.9
3           4     Le Thanh Huyen  Toan Tin         20    7.2
4           5        Vu Quoc Anh  May Tinh         24    8.0
5           6     Dang Thuy Linh  May Tinh         24    5.5
6           7      Bui Tien Dung   Kinh Te         34    9.2
7           8        Ho Ngoc Mai  Toan Tin         20    8.8
8           9     Duong Huu Phuc   Kinh Te         26    7.2
9          10       Cao Thi Hanh  May Tinh         26    7.0


Loại bỏ các bản ghi có course_id không tồn tại trong bảng course:
   student_id               name     class  course_id  score
0           1  Nguyen Minh Hoang  May Tinh         12    6.7
1           2       Tran Thi Lan   Kinh Te         34    9.2
2           3       Pham Van Nam  To

**Kết luận**

- Các giá trị course_id còn thiếu đã được cập nhật bằng các mã môn học hợp lệ từ bảng course.
- Các bản ghi có course_id không tồn tại trong bảng course đã bị loại bỏ.
- Sau khi xử lý, dữ liệu trong bảng student chỉ còn các bản ghi hợp lệ và không còn giá trị bị thiếu.
- Bảng dữ liệu sau khi làm sạch đảm bảo tính nhất quán giữa hai bảng student và course, sẵn sàng cho các bước phân tích tiếp theo.

##a. Tổng số sinh viên, điểm trung bình của từng lớp.

In [ ]:
print("Thống kê theo lớp:")

cursor.execute('''
SELECT
    class,
    COUNT(*) AS total_students,
    ROUND(AVG(score), 2) AS avg_score
FROM student
GROUP BY class;
''')

df_class = pd.DataFrame(cursor.fetchall(), columns=[
    'class', 'total_students', 'avg_score'
])

print(df_class)
print("\n")

Thống kê theo lớp:
      class  total_students  avg_score
0   Kinh Te               3       8.53
1  May Tinh               2       6.85
2  Toan Tin               1       7.90




**Kết luận**

- Lớp Kinh Te có số lượng sinh viên nhiều nhất (3 sinh viên) và điểm trung bình cao nhất (8.53).
- Lớp May Tinh có 2 sinh viên với điểm trung bình là 6.85.
- Lớp Toan Tin có ít sinh viên nhất (1 sinh viên) với điểm trung bình là 7.90.
- Nhìn chung, lớp Kinh Te có kết quả học tập tốt nhất trong các lớp.

##b. Tổng số sinh viên, điểm trung bình của từng môn học.

In [ ]:
print("Thống kê theo môn học:")

cursor.execute('''
SELECT
    c.course_name,
    COUNT(s.student_id) AS total_students,
    ROUND(AVG(s.score), 2) AS avg_score
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
GROUP BY c.course_name;
''')

df_course = pd.DataFrame(cursor.fetchall(), columns=[
    'course_name', 'total_students', 'avg_score'
])

print(df_course)
print("\n")

Thống kê theo môn học:
  course_name  total_students  avg_score
0   Giai tich               1       6.70
1    Thong ke               2       9.20
2     Tin hoc               3       7.37




**Kết luận**

- Môn Tin hoc có số lượng sinh viên nhiều nhất (3 sinh viên) với điểm trung bình là 7.37.
- Môn Thong ke có 2 sinh viên và có điểm trung bình cao nhất (9.20).
- Môn Giai tich có ít sinh viên nhất (1 sinh viên) với điểm trung bình là 6.70.
- Nhìn chung, môn Thong ke có kết quả học tập tốt nhất, trong khi Giai tich có kết quả thấp hơn.

## c. Phân loại thi đua theo số điểm của từng môn học biết:

- i. Điểm TB ≥ 9.0: Xuất sắc.
- ii. 6.0 ≤ Điểm TB ≤ 8.9: Tốt.
- iii. Điểm TB < 6.0: Kém.

In [ ]:
print("Phân loại thi đua theo môn học:")

cursor.execute('''
SELECT
    c.course_name,
    ROUND(AVG(s.score), 2) AS avg_score,
    CASE
        WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
        WHEN AVG(s.score) >= 6.0 THEN 'Tot'
        ELSE 'Kem'
    END AS classification
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
GROUP BY c.course_name;
''')

df_classification = pd.DataFrame(cursor.fetchall(), columns=[
    'course_name', 'avg_score', 'classification'
])

print(df_classification)
print("\n")

Phân loại thi đua theo môn học:
  course_name  avg_score classification
0   Giai tich       6.70            Tot
1    Thong ke       9.20       Xuat sac
2     Tin hoc       7.37            Tot




**Nhận xét**

- Môn Thong ke được xếp loại Xuất sắc do có điểm trung bình cao nhất (9.20).
- Các môn Giai tich và Tin hoc được xếp loại Tốt vì có điểm trung bình nằm trong khoảng từ 6.0 đến dưới 9.0.
- Không có môn học nào bị xếp loại Kém.
- Nhìn chung, kết quả học tập ở các môn đều đạt mức từ Tốt trở lên.

# Câu 3. Hãy xếp hạng sinh viên thông qua:

 a. Điểm số.

 b. Điểm số theo lớp học.

 c. Điểm số theo mã môn học.

và cho biết top 3 sinh viện đạt thứ hạng cao nhất, top 3 sinh viên đạt thứ hạng thấp nhất theo
từng trường hợp trên.

###a. Điểm số

In [ ]:
#Truy vấn điểm số
print("Xếp hạng sinh viên theo điểm số toàn bộ:")

cursor.execute('''
SELECT
    student_id,
    name,
    class,
    course_id,
    score,
    RANK() OVER (ORDER BY score DESC) AS rank_score_desc
FROM student;
''')

df_rank_all = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_rank_all)
print("\n")

Xếp hạng sinh viên theo điểm số toàn bộ:
   student_id               name     class  course_id  score  rank
0           2       Tran Thi Lan   Kinh Te         34    9.2     1
1           7      Bui Tien Dung   Kinh Te         34    9.2     1
2           3       Pham Van Nam  Toan Tin         26    7.9     3
3           9     Duong Huu Phuc   Kinh Te         26    7.2     4
4          10       Cao Thi Hanh  May Tinh         26    7.0     5
5           1  Nguyen Minh Hoang  May Tinh         12    6.7     6




**Kết luận**

- Lớp Kinh Te có thành tích nổi bật nhất với cả hạng 1 và hạng 4, thể hiện chất lượng học tập ổn định.
- Lớp May Tinh có xu hướng điểm thấp hơn so với các lớp khác, cần xem xét cải thiện phương pháp giảng dạy.
- Không có sinh viên dưới 6.5 điểm, thể hiện mặt bằng học lực khá tốt.

In [ ]:
print("Top 3 sinh viên có thứ hạng cao nhất theo điểm số toàn bộ:")

cursor.execute('''
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (ORDER BY score DESC) AS rank_score_desc
    FROM student
)
WHERE rank_score_desc <= 3;
''')

df_top3_all_high = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_top3_all_high)
print("\n")

Top 3 sinh viên có thứ hạng cao nhất theo điểm số toàn bộ:
   student_id           name     class  course_id  score  rank
0           2   Tran Thi Lan   Kinh Te         34    9.2     1
1           7  Bui Tien Dung   Kinh Te         34    9.2     1
2           3   Pham Van Nam  Toan Tin         26    7.9     3




Top 3 cao nhất:
- Trần Thị Lan và Bùi Tiến Dũng (9.2 điểm) — Hạng 1.
- Phạm Văn Nam (7.9 điểm) — Hạng 3.


In [ ]:
print("Top 3 sinh viên có thứ hạng thấp nhất theo điểm số toàn bộ:")

cursor.execute('''
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (ORDER BY score ASC) AS rank_score_asc
    FROM student
)
WHERE rank_score_asc <= 3;
''')

df_top3_all_low = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_top3_all_low)
print("\n")

Top 3 sinh viên có thứ hạng thấp nhất theo điểm số toàn bộ:
   student_id               name     class  course_id  score  rank
0           1  Nguyen Minh Hoang  May Tinh         12    6.7     1
1          10       Cao Thi Hanh  May Tinh         26    7.0     2
2           9     Duong Huu Phuc   Kinh Te         26    7.2     3




Top 3 thấp nhất:
- Nguyễn Minh Hoàng (6.7 điểm) — Hạng 6 (thấp nhất).
- Cao Thị Hạnh (7.0 điểm) — Hạng 5.
- Dương Hữu Phúc (7.2 điểm) — Hạng 4.

##b. Điểm số theo lớp học.

In [ ]:
print("Xếp hạng sinh viên theo điểm số trong từng lớp:")

cursor.execute('''
SELECT
    student_id,
    name,
    class,
    course_id,
    score,
    RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
FROM student;
''')

df_rank_class = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_rank_class)
print("\n")

Xếp hạng sinh viên theo điểm số trong từng lớp:
   student_id               name     class  course_id  score  rank
0           2       Tran Thi Lan   Kinh Te         34    9.2     1
1           7      Bui Tien Dung   Kinh Te         34    9.2     1
2           9     Duong Huu Phuc   Kinh Te         26    7.2     3
3          10       Cao Thi Hanh  May Tinh         26    7.0     1
4           1  Nguyen Minh Hoang  May Tinh         12    6.7     2
5           3       Pham Van Nam  Toan Tin         26    7.9     1




**Kết luận:**
- Lớp Kinh Te có sự cạnh tranh cao với khoảng cách điểm lớn giữa các hạng.
- Lớp May Tinh cần nâng cao chất lượng học tập do điểm số dẫn đầu chưa thực sự nổi bật.
- Lớp Toan Tin có mặt bằng điểm khá nhưng thiếu sự cạnh tranh, cần thu hút thêm sinh viên.

In [ ]:
print("Top 3 sinh viên có thứ hạng cao nhất theo từng lớp:")

cursor.execute('''
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3;
''')

df_top3_class_high = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_top3_class_high)
print("\n")

Top 3 sinh viên có thứ hạng cao nhất theo từng lớp:
   student_id               name     class  course_id  score  rank
0           2       Tran Thi Lan   Kinh Te         34    9.2     1
1           7      Bui Tien Dung   Kinh Te         34    9.2     1
2           9     Duong Huu Phuc   Kinh Te         26    7.2     3
3          10       Cao Thi Hanh  May Tinh         26    7.0     1
4           1  Nguyen Minh Hoang  May Tinh         12    6.7     2
5           3       Pham Van Nam  Toan Tin         26    7.9     1




In [ ]:
print("Top 3 sinh viên có thứ hạng thấp nhất theo từng lớp:")

cursor.execute('''
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class
    FROM student
)
WHERE rank_in_class <= 3;
''')

df_top3_class_low = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_top3_class_low)
print("\n")

Top 3 sinh viên có thứ hạng thấp nhất theo từng lớp:
   student_id               name     class  course_id  score  rank
0           9     Duong Huu Phuc   Kinh Te         26    7.2     1
1           2       Tran Thi Lan   Kinh Te         34    9.2     2
2           7      Bui Tien Dung   Kinh Te         34    9.2     2
3           1  Nguyen Minh Hoang  May Tinh         12    6.7     1
4          10       Cao Thi Hanh  May Tinh         26    7.0     2
5           3       Pham Van Nam  Toan Tin         26    7.9     1




 Top 3 cao nhất & thấp nhất theo điểm số từng lớp

Lớp Kinh Tế:
- Cao nhất: Trần Thị Lan và Bùi Tiến Dũng (9.2 điểm) — Đồng hạng 1.
- Thấp nhất: Dương Hữu Phúc (7.2 điểm) — Hạng 3.

Lớp Máy Tính:
- Cao nhất: Cao Thị Hạnh (7.0 điểm) — Hạng 1.
- Thấp nhất: Nguyễn Minh Hoàng (6.7 điểm) — Hạng 2 (cũng là thấp nhất).

Lớp Toán Tin:
- Cao nhất: Phạm Văn Nam (7.9 điểm) — Hạng 1.
- Thấp nhất: Cũng là Phạm Văn Nam (7.9 điểm) — Hạng 1 (vì chỉ có 1 sinh viên).

##c. Xếp hạng sinh viên theo điểm số trong từng mã môn học

In [ ]:
print("Xếp hạng sinh viên theo điểm số trong từng mã môn học:")

cursor.execute('''
SELECT
    student_id,
    name,
    class,
    course_id,
    score,
    RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
FROM student;
''')

df_rank_course = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_rank_course)
print("\n")

Xếp hạng sinh viên theo điểm số trong từng mã môn học:
   student_id               name     class  course_id  score  rank
0           1  Nguyen Minh Hoang  May Tinh         12    6.7     1
1           3       Pham Van Nam  Toan Tin         26    7.9     1
2           9     Duong Huu Phuc   Kinh Te         26    7.2     2
3          10       Cao Thi Hanh  May Tinh         26    7.0     3
4           2       Tran Thi Lan   Kinh Te         34    9.2     1
5           7      Bui Tien Dung   Kinh Te         34    9.2     1




**Kết luận:**
- Môn Giai tich: Nguyễn Minh Hoàng là sinh viên duy nhất với 6.7 điểm (mức trung bình), cần cải thiện thêm.
- Môn Tin hoc: Phạm Văn Nam dẫn đầu với 7.9 điểm; Dương Hữu Phúc và Cao Thị Hạnh lần lượt đạt 7.2 và 7.0 điểm. Điểm trung bình khá ổn, cần cố gắng hơn để đạt xuất sắc.
- Môn Thong ke: Trần Thị Lan và Bùi Tiến Dũng cùng đạt 9.2 điểm (xuất sắc), có điểm trung bình cao nhất, thể hiện chất lượng học tập tốt.

In [ ]:
print("Top 3 sinh viên có thứ hạng cao nhất theo từng mã môn học:")

cursor.execute('''
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course
    FROM student
)
WHERE rank_in_course <= 3;
''')

df_top3_course_high = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_top3_course_high)
print("\n")

Top 3 sinh viên có thứ hạng cao nhất theo từng mã môn học:
   student_id               name     class  course_id  score  rank
0           1  Nguyen Minh Hoang  May Tinh         12    6.7     1
1           3       Pham Van Nam  Toan Tin         26    7.9     1
2           9     Duong Huu Phuc   Kinh Te         26    7.2     2
3          10       Cao Thi Hanh  May Tinh         26    7.0     3
4           2       Tran Thi Lan   Kinh Te         34    9.2     1
5           7      Bui Tien Dung   Kinh Te         34    9.2     1




In [ ]:
print("Top 3 sinh viên có thứ hạng thấp nhất theo từng mã môn học:")

cursor.execute('''
SELECT *
FROM (
    SELECT
        student_id,
        name,
        class,
        course_id,
        score,
        RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank_in_course
    FROM student
)
WHERE rank_in_course <= 3;
''')

df_top3_course_low = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'rank']
)

print(df_top3_course_low)
print("\n")

Top 3 sinh viên có thứ hạng thấp nhất theo từng mã môn học:
   student_id               name     class  course_id  score  rank
0           1  Nguyen Minh Hoang  May Tinh         12    6.7     1
1          10       Cao Thi Hanh  May Tinh         26    7.0     1
2           9     Duong Huu Phuc   Kinh Te         26    7.2     2
3           3       Pham Van Nam  Toan Tin         26    7.9     3
4           2       Tran Thi Lan   Kinh Te         34    9.2     1
5           7      Bui Tien Dung   Kinh Te         34    9.2     1




Top 3 cao nhất & thấp nhất theo điểm số từng môn học

Môn Giải tích:
- Cao nhất: Nguyễn Minh Hoàng (6.7 điểm) — Hạng 1.
- Thấp nhất: Cũng là Nguyễn Minh Hoàng (6.7 điểm) — Hạng 1 vì chỉ có 1 sinh viên.

Môn Tin học:
- Cao nhất: Phạm Văn Nam (7.9 điểm) — Hạng 1.
- Thấp nhất: Cao Thị Hạnh (7.0 điểm) — Hạng 3.

Môn Thống kê:
- Cao nhất: Trần Thị Lan và Bùi Tiến Dũng (9.2 điểm) — Đồng hạng 1.
- Thấp nhất: Cũng là Trần Thị Lan và Bùi Tiến Dũng (9.2 điểm) — Đồng hạng 1 do điểm số bằng nhau.

#Câu 4. Hãy bổ sung thêm một trường graduation_date có kiểu dữ liệu là DATETIME vào bảng student để xác định thời gian tốt nghiệp của từng bạn, trong đó thời gian tốt nghiệp được xác định bởi thời gian hiện tại cộng với số hạng tương ứng của bạn đó tính theo điểm số.

## Bước 1. Thêm cột graduation_date

In [ ]:
print("Thêm cột graduation_date vào bảng student:")

cursor.execute('''
ALTER TABLE student
ADD COLUMN graduation_date DATETIME;
''')
conn.commit()

print("Đã thêm cột graduation_date.")
print("\n")

Thêm cột graduation_date vào bảng student:
Đã thêm cột graduation_date.




##Bước 2. Cập nhật graduation_date theo thứ hạng điểm số

In [ ]:
print("Cập nhật graduation_date theo thứ hạng điểm số:")

cursor.execute('''
WITH ranked AS (
    SELECT
        student_id,
        RANK() OVER (ORDER BY score DESC) AS rank_score
    FROM student
)
UPDATE student
SET graduation_date = datetime(
    'now',
    '+' || (
        SELECT rank_score
        FROM ranked
        WHERE ranked.student_id = student.student_id
    ) || ' days'
);
''')
conn.commit()

print("Đã cập nhật graduation_date.")
print("\n")

Cập nhật graduation_date theo thứ hạng điểm số:
Đã cập nhật graduation_date.




##Bước 3. Bảng student sau khi cập nhật graduation_date

In [ ]:
print("Bảng student sau khi cập nhật graduation_date:")

cursor.execute('''
SELECT
    student_id,
    name,
    class,
    course_id,
    score,
    graduation_date
FROM student
ORDER BY score DESC;
''')

df_graduation = pd.DataFrame(
    cursor.fetchall(),
    columns=['student_id', 'name', 'class', 'course_id', 'score', 'graduation_date']
)

print(df_graduation)
print("\n")

Bảng student sau khi cập nhật graduation_date:
   student_id               name     class  course_id  score      graduation_date
0           2       Tran Thi Lan   Kinh Te         34    9.2  2026-04-03 09:27:00
1           7      Bui Tien Dung   Kinh Te         34    9.2  2026-04-03 09:27:00
2           3       Pham Van Nam  Toan Tin         26    7.9  2026-04-05 09:27:00
3           9     Duong Huu Phuc   Kinh Te         26    7.2  2026-04-06 09:27:00
4          10       Cao Thi Hanh  May Tinh         26    7.0  2026-04-07 09:27:00
5           1  Nguyen Minh Hoang  May Tinh         12    6.7  2026-04-08 09:27:00




**Kết luận**

- Hai sinh viên Trần Thị Lan và Bùi Tiến Dũng có điểm cao nhất (9.2) nên được xếp hạng 1 và có thời gian tốt nghiệp sớm nhất vào ngày 03/04/2026.

- Sinh viên Phạm Văn Nam có điểm 7.9, xếp hạng tiếp theo và dự kiến tốt nghiệp vào ngày 05/04/2026.

- Sinh viên Dương Hữu Phúc có điểm 7.2, tốt nghiệp vào ngày 06/04/2026.

- Sinh viên Cao Thị Hạnh có điểm 7.0, tốt nghiệp vào ngày 07/04/2026.

- Sinh viên Nguyễn Minh Hoàng có điểm thấp nhất (6.7) nên có thời gian tốt nghiệp muộn nhất vào ngày 08/04/2026.

- Nhìn chung, sinh viên có điểm càng cao thì thời gian tốt nghiệp càng sớm, đúng với yêu cầu của bài toán.